# Análisis estadístico de los modelos predictores del BIS

En esta libreta se comparan estadísticamente los tres modelos de predicción del BIS evaluados en `Comparacion_Modelos_BIS.ipynb`: el modelo PK-PD, la red LSTM de referencia y el modelo híbrido.


**Procedimiento.**
1. Estadística descriptiva (media, desviación estándar y mediana) por modelo.
2. Prueba de Friedman para contrastar, de manera global, si al menos un modelo difiere de los demás.
3. Prueba de rangos con signo de Wilcoxon para cada par de modelos, con la corrección de Holm para comparaciones múltiples.



## Etapa 1: Configuración y carga de datos

In [1]:
# =============================================================================
# CONFIGURACIÓN
# =============================================================================
import numpy as np
import pandas as pd
from scipy import stats

RUTA_METRICAS = "Metricas_Modelos_por_Paciente.csv"   # Métricas por paciente y modelo
ALFA          = 0.05                                    # Nivel de significancia
METRICAS      = ["MAE", "MSE", "RMSE", "MAPE"]
MODELOS       = ["PK-PD", "LSTM (LDS)", "Modelo híbrido"]

df = pd.read_csv(RUTA_METRICAS)
print(f"[INFO] Registros cargados: {len(df)}  "
      f"({df['Caso_VitalDB'].nunique()} pacientes × {df['Modelo'].nunique()} modelos)")

[INFO] Registros cargados: 96  (32 pacientes × 3 modelos)


## Etapa 2: Estadística descriptiva

In [2]:
# =============================================================================
# ESTADÍSTICA DESCRIPTIVA POR MODELO
# =============================================================================
resumen = (df.groupby("Modelo")[METRICAS]
             .agg(["mean", "std", "median"])
             .reindex(MODELOS)
             .round(3))
resumen

MAE                      MSE                      RMSE  \
                  mean     std  median     mean      std   median    mean   
Modelo                                                                      
PK-PD           18.693  10.261  16.043  567.597  711.563  368.119  21.795   
LSTM (LDS)       9.468   3.354   8.520  150.189   96.960  121.814  11.747   
Modelo híbrido   8.236   2.551   7.808  115.573   59.819  106.783  10.405   

                                 MAPE                  
                  std  median    mean     std  median  
Modelo                                                 
PK-PD           9.774  19.186  47.390  37.929  35.060  
LSTM (LDS)      3.547  11.036  24.398  13.947  20.345  
Modelo híbrido  2.748  10.333  20.454   9.480  18.070

## Etapa 3: Prueba de Friedman y comparaciones por pares

In [3]:
# =============================================================================
# PRUEBA DE FRIEDMAN Y PRUEBAS DE WILCOXON CON CORRECCIÓN DE HOLM
# =============================================================================
# Los tres modelos se evaluaron sobre los mismos pacientes, por lo que el diseño
# es apareado. La prueba de Friedman contrasta la igualdad global de los tres
# modelos; las comparaciones por pares se efectúan con la prueba de rangos con
# signo de Wilcoxon y los valores-p se ajustan mediante el método de Holm.

def correccion_holm(valores_p):
    """Devuelve los valores-p ajustados mediante el método secuencial de Holm."""
    valores_p = np.asarray(valores_p, dtype=float)
    m = len(valores_p)
    orden = np.argsort(valores_p)
    ajustados = np.empty(m)
    acumulado = 0.0
    for rango, idx in enumerate(orden):
        acumulado = max(acumulado, (m - rango) * valores_p[idx])
        ajustados[idx] = min(1.0, acumulado)
    return ajustados


PARES = [("Modelo híbrido", "LSTM (LDS)"),
         ("Modelo híbrido", "PK-PD"),
         ("LSTM (LDS)", "PK-PD")]

filas = []
for metrica in METRICAS:
    tabla = df.pivot(index="Caso_VitalDB", columns="Modelo", values=metrica)[MODELOS]
    chi2, p_friedman = stats.friedmanchisquare(*[tabla[m] for m in MODELOS])
    print(f"{metrica:5s} | Friedman: chi2 = {chi2:.2f}   valor-p = {p_friedman:.2e}")

    valores_p = []
    parciales = []
    for a, b in PARES:
        prueba = stats.wilcoxon(tabla[a], tabla[b])
        dif = tabla[a] - tabla[b]
        valores_p.append(prueba.pvalue)
        parciales.append(dict(Metrica=metrica, Comparacion=f"{a} frente a {b}",
                              Diferencia_mediana=round(float(np.median(dif)), 3),
                              Pacientes_primero_mejor=f"{int((dif < 0).sum())}/{len(dif)}",
                              Estadistico=float(prueba.statistic),
                              Valor_p=prueba.pvalue))
    for fila, p_holm in zip(parciales, correccion_holm(valores_p)):
        fila["Valor_p_Holm"] = p_holm
        fila["Significativa"] = p_holm < ALFA
        filas.append(fila)

comparaciones = pd.DataFrame(filas)
comparaciones

MAE   | Friedman: chi2 = 43.94   valor-p = 2.88e-10
MSE   | Friedman: chi2 = 49.75   valor-p = 1.57e-11
RMSE  | Friedman: chi2 = 49.75   valor-p = 1.57e-11
MAPE  | Friedman: chi2 = 43.94   valor-p = 2.88e-10


,Metrica,Comparacion,Diferencia_mediana,Pacientes_primero_mejor,Estadistico,Valor_p,Valor_p_Holm,Significativa
0,MAE,Modelo híbrido frente a LSTM (LDS),-1.190,25/32,81.0,3.345464e-04,3.345464e-04,True
1,MAE,Modelo híbrido frente a PK-PD,-7.511,30/32,6.0,6.519258e-09,1.955777e-08,True
2,MAE,LSTM (LDS) frente a PK-PD,-6.636,31/32,18.0,1.178123e-07,2.356246e-07,True
3,MSE,Modelo híbrido frente a LSTM (LDS),-22.199,27/32,76.0,2.138605e-04,2.138605e-04,True
4,MSE,Modelo híbrido frente a PK-PD,-264.215,31/32,1.0,9.313226e-10,2.793968e-09,True
5,MSE,LSTM (LDS) frente a PK-PD,-245.760,31/32,12.0,3.259629e-08,6.519258e-08,True
6,RMSE,Modelo híbrido frente a LSTM (LDS),-1.019,27/32,77.0,4.709133e-04,4.709133e-04,True
7,RMSE,Modelo híbrido frente a PK-PD,-8.916,31/32,1.0,9.313226e-10,2.793968e-09,True
8,RMSE,LSTM (LDS) frente a PK-PD,-9.031,31/32,8.0,1.164153e-08,2.328306e-08,True
9,MAPE,Modelo híbrido frente a LSTM (LDS),-3.410,25/32,71.0,3.074101e-04,3.074101e-04,True
